# Semantic Image Search Over an Instagram Profile

Search a public Instagram profile's images with plain language, "a dog wearing sunglasses" or "horseback riding", instead of keywords. The pipeline:

1. Fetch posts from a public profile with [SerpApi's Instagram Profile API](https://serpapi.com/instagram-profile-api), paging through `next_page_token` to pull plenty of images for good results.
2. Download each post's cover image, photos and video thumbnails, so the whole feed is searchable (great for video-heavy profiles).
3. Embed each image with [Jina AI's `jina-embeddings-v5-omni-small`](https://jina.ai/models/jina-embeddings-v5-omni-small/) (a shared text↔image space) via the [Jina Embeddings API](https://jina.ai/embeddings/).
4. Index the vectors + metadata in an Elasticsearch [`dense_vector`](https://www.elastic.co/docs/solutions/search/vector/dense-vector) index.
5. Query in plain language: embed the text into the same space and retrieve matches with Elasticsearch [kNN](https://www.elastic.co/docs/reference/query-languages/query-dsl/query-dsl-knn-query).

Because we embed video covers too, a hit on a video is a nice proof that we're matching the image, not the caption. The same code runs against local Docker (`start-local`) or [Elastic Cloud Serverless](https://www.elastic.co/cloud/serverless), only `.env` differs. We store only vectors + metadata (caption, post URL, image/thumbnail URL, likes/comments), never raw image bytes; images live in a local `images/` folder just long enough to embed and display results.

## Step 1: Setup

Install dependencies, load secrets from `.env`, and set `PROFILE_USERNAME` / `QUERY`.

Where Elasticsearch runs, pick one (the notebook code is identical, only `.env` changes):
- Local Docker (needs a few GB RAM): `curl -fsSL https://elastic.co/start-local | sh` → set `ES_URL=http://localhost:9200` and `ES_API_KEY=<ES_LOCAL_API_KEY>` (printed by the script).
- [Elastic Cloud Serverless](https://www.elastic.co/cloud/serverless) (no local compute, good if your machine is light): create a free project, then set `ES_URL=<your endpoint>` and `ES_API_KEY=<your key>`.

The [Jina embedding model](https://jina.ai/embeddings/) always runs in Jina's cloud (we call the API), so it never loads on your machine either way.


In [1]:
# Install dependencies (safe to re-run; -q keeps the output quiet).
%pip install -q serpapi requests pillow python-dotenv elasticsearch

Note: you may need to restart the kernel to use updated packages.


### Config: The Two Knobs You'll Swap

`PROFILE_USERNAME` and `QUERY` are one-line changes. Re-run from Step 2 after changing the profile, or just the query cell in Step 5 to try a new search against an already-indexed profile.


In [2]:
# ── Swap these for trial-and-error ───────────────────────────────────
PROFILE_USERNAME = "apple"                       # any *public* IG profile (try "nvidia", "nasa")
QUERY            = "a dog wearing sunglasses"    # plain-language search over the images

# More queries to try once indexed. Just reassign QUERY (or call search(...)) and re-run Step 5:
#   "snow-capped mountains"        "horseback riding"                 "a close-up portrait"
#   "a cat"                        "a person silhouetted at sunset"   "someone holding a phone"

# ── Pipeline knobs (sane defaults; rarely need changing) ─────────────
# We embed EVERY post's cover image (photos *and* video thumbnails), so the whole feed is
# searchable (key for video-heavy profiles like @apple). Each page bills ~1 SerpApi credit.
MAX_ITEMS   = 1500                             # cap on cover images to collect (Step 2)
MAX_PAGES   = 100                              # pages to walk (~12 posts/page → ~1 credit each)

JINA_MODEL  = "jina-embeddings-v5-omni-small"  # multimodal model (shared text/image space)
EMBED_DIM   = 1024                             # jina-embeddings-v5-omni-small output dimension
EMBED_BATCH = 8                                # images per embed request

ES_INDEX    = "instagram_photos"               # Elasticsearch index name
IMAGES_DIR  = "images"                         # local scratch folder for downloaded images

print(f"@{PROFILE_USERNAME} | {QUERY!r} | {JINA_MODEL} ({EMBED_DIM}d)")

@apple | 'a dog wearing sunglasses' | jina-embeddings-v5-omni-small (1024d)


### Secrets

All keys live in a `.env` file next to this notebook (copy `.env.example` → `.env` and fill it in). Nothing is hardcoded.

```
SERPAPI_API_KEY=...
JINA_API_KEY=...
ES_URL=http://localhost:9200
ES_API_KEY=...        # the ES_LOCAL_API_KEY printed by start-local
```


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads ./.env into the environment

SERPAPI_API_KEY = os.getenv("SERPAPI_API_KEY")
JINA_API_KEY    = os.getenv("JINA_API_KEY")
ES_URL          = os.getenv("ES_URL", "http://localhost:9200")
ES_API_KEY      = os.getenv("ES_API_KEY")

# Quick presence check (masked) so a missing key fails loudly here, not deep in a call.
for name, value in {
    "SERPAPI_API_KEY": SERPAPI_API_KEY,
    "JINA_API_KEY":    JINA_API_KEY,
    "ES_URL":          ES_URL,
    "ES_API_KEY":      ES_API_KEY,
}.items():
    status = "set ✓" if value else "MISSING ✗"
    print(f"{name:<16} {status}")

SERPAPI_API_KEY  set ✓
JINA_API_KEY     set ✓
ES_URL           set ✓
ES_API_KEY       set ✓


In [4]:
# Initialize the SerpApi client, the Jina request settings, and the Elasticsearch client.
import requests
import serpapi
from elasticsearch import Elasticsearch

serp_client = serpapi.Client(api_key=SERPAPI_API_KEY, timeout=30)

JINA_URL = "https://api.jina.ai/v1/embeddings"
jina_headers = {"Authorization": f"Bearer {JINA_API_KEY}", "Content-Type": "application/json"}

es = Elasticsearch(ES_URL, api_key=ES_API_KEY, request_timeout=60, max_retries=3, retry_on_timeout=True)
print("Elasticsearch:", "connected ✓" if es.ping() else "NOT reachable ✗ (is start-local running?)")
print("SerpApi + Jina ready.")

Elasticsearch: connected ✓
SerpApi + Jina ready.


Run Step 1. You should see your keys `set ✓` and `Elasticsearch: connected ✓` (start `start-local` first, see the README). Next we fetch the profile's posts.


## Step 2: Fetch the Profile's Posts

The [Instagram Profile API](https://serpapi.com/instagram-profile-api) takes `engine="instagram_profile"` and `profile_id` (the username). Each response returns a batch of ~12 posts in `profile_results.posts` plus a `serpapi_pagination.next_page_token` we feed back in to page through the feed.

We embed every post's cover image, photos and video thumbnails, so the whole feed is searchable. That matters for video-heavy profiles like `@apple` (~65% reels): a video's `serpapi_display_url` is its poster frame, and searching it is a great demo that we're matching the image, not the caption. We page until we hit `MAX_ITEMS`, capped at `MAX_PAGES`.

Incremental by default. We use each post's `shortcode` as its Elasticsearch document id, so before fetching we look up which posts are already indexed for this profile. Because the feed is newest-first, once we hit a page with no new posts we've caught up and stop, so re-running the same profile costs only a page or two and pulls just the latest posts (saving [SerpApi](https://serpapi.com/) credits, Jina tokens, and disk).

The Instagram feed is a little flaky when you page deep, so two habits keep the fetch robust:
- **Retry transient failures.** A page can intermittently come back with an error or empty results, so we retry a page before giving up. (We do *not* set `no_cache`: we tested it and it makes no measurable difference to how far pagination gets. The run-to-run variance from feed flakiness dwarfs any cache effect, so we skip it, which also lets a re-run reuse SerpApi's cache instead of re-billing every page.)
- **Surface `results["error"]`.** Private or embeds-disabled profiles (and dead tokens) come back as an error or `is_private` / `is_embeds_disabled`; we stop gracefully and say why instead of pretending we're done.

Confirmed from the live response: `media_captions` is a list of plain strings (`media_captions[0]` is the caption), and there's no post timestamp, so we store the metadata we do have (caption, likes, comments, `is_video`). One gotcha on **likes**: `liked_by_count` is only populated for the newest ~page of posts, while `media_preview_likes_count` is returned for **every** post (and is identical where both exist), so in Step 3 we read `liked_by_count or media_preview_likes_count` to get a like count for the whole feed.

In [5]:
def get_caption(post):
    """media_captions is a list of plain strings; return the first (or '')."""
    caps = post.get("media_captions") or []
    return caps[0] if caps else ""


def existing_shortcodes(username):
    """Shortcodes already indexed for this profile (empty set if the index doesn't exist yet)."""
    if not es.indices.exists(index=ES_INDEX):
        return set()
    resp = es.search(
        index=ES_INDEX, size=10_000,
        query={"term": {"username": username}},
        source_includes=["shortcode"],
    )
    return {h["_source"]["shortcode"] for h in resp["hits"]["hits"]}


def fetch_profile_posts(username, max_items=MAX_ITEMS, max_pages=MAX_PAGES, known=None):
    """Page newest-first, keeping only posts we don't already have. Instagram's feed is
    reverse-chronological, so once a whole page is already indexed we've caught up and stop.
    re-running a profile then costs a page or two and pulls only the latest posts.

    Two layers of dedup: against `known` (already in Elasticsearch → incremental re-runs) and
    against what we've already collected this run (the live feed shifts between page scrapes,
    so the same post can reappear across pages, and we embed each only once).
    """
    known = set(known or ())
    base = {"engine": "instagram_profile", "profile_id": username}
    params = dict(base)
    profile, kept, seen = {}, [], set()

    for page in range(1, max_pages + 1):
        results = serp_client.search(params)
        # Empty/error pages can be transient, so retry the same page once before giving up.
        if results.get("error"):
            results = serp_client.search(params)
        if results.get("error"):
            print(f"Page {page}: stopped: {results['error']}")
            break

        profile = results.get("profile_results", {})
        if profile.get("is_private") or profile.get("is_embeds_disabled"):
            print(f"@{username} has no embeddable media (private or embeds disabled), stopping.")
            break

        batch = profile.get("posts", [])
        new_vs_known = [p for p in batch if p.get("shortcode") not in known]
        # Keep only ones not already collected this run (drops cross-page feed overlap).
        fresh = [p for p in new_vs_known if p.get("shortcode") not in seen]
        seen.update(p["shortcode"] for p in fresh)
        kept.extend(fresh)
        vids = sum(1 for p in fresh if p.get("is_video"))
        print(f"Page {page}: {len(batch)} posts, +{len(fresh)} new (+{vids} video), total {len(kept)}")

        # A full page with nothing new *vs the index* means we've reached already-indexed posts.
        if batch and not new_vs_known:
            print("Caught up to already-indexed posts, stopping (incremental).")
            break

        next_token = results.get("serpapi_pagination", {}).get("next_page_token")
        if not next_token or len(kept) >= max_items:
            break
        # Keep the base params and swap in the fresh token (token-only would 400).
        params = {**base, "next_page_token": next_token}

    return profile, kept[:max_items]

In [6]:
known = existing_shortcodes(PROFILE_USERNAME)
if known:
    print(f"@{PROFILE_USERNAME}: {len(known)} posts already indexed, fetching only newer ones.\n")

profile, posts = fetch_profile_posts(PROFILE_USERNAME, known=known)
vids = sum(1 for p in posts if p.get("is_video"))
print(f"\nCollected {len(posts)} new posts from @{PROFILE_USERNAME}: {profile.get('full_name', '')}"
      f"  ({len(posts) - vids} photos, {vids} video covers)")

@apple: 579 posts already indexed, fetching only newer ones.



Page 1: 12 posts, +1 new (+0 video), total 1


Page 2: 12 posts, +0 new (+0 video), total 1
Caught up to already-indexed posts, stopping (incremental).

Collected 1 new posts from @apple: apple  (1 photos, 0 video covers)


In [7]:
# Sanity-check what we collected: caption + cover image + post type for the first 5 posts.
for p in posts[:5]:
    caption = " ".join(get_caption(p).split())[:80]
    kind = "video" if p.get("is_video") else "photo"
    # liked_by_count is only set on the newest ~page; media_preview_likes_count covers all posts.
    likes = p.get("liked_by_count") or p.get("media_preview_likes_count")
    print(f"[{kind}] https://www.instagram.com/p/{p.get('shortcode')}/")
    print(f"   caption: {caption!r}")
    print(f"   image  : {p.get('serpapi_display_url') or p.get('display_url')}")
    print(f"   likes  : {likes}  comments: {p.get('comments_count')}")

print(f"\nNew cover images ready to embed: {len(posts)}")

[photo] https://www.instagram.com/p/DaS-JrkFElP/
   caption: 'Bring your own biome. #ShotoniPhone by @danielidle'
   image  : https://serpapi.com/images/url/JxX0L3ichY7PjpswGMTVl-FmsMEGUsmqYDdLYBN2CZuswgWBITh_sEltkjZv2WufpjTqvd9hNNKM5vf9_vKLaz2or5almBS6FRrUUhGATNaIg1C66r5Xvclkb10tTZDp257vAUQsz0E2Qgj6JfKxgx3oIAzRzEe-WzrQtiEkhLgIEgciDzmlMI9D903pgTZKg8mXrUPKYRqAP_5KqbhtTmtau0YpWMk1_d9Ljx6rNEUPJxnNXFbY3eLnLuXdbL8ZXwHA47yQy6N6iYEOd-MmyXfkY34puHi9XPZyO8Z99ryI1vUxl_regCJOzs9JGuWrsL2-xGHtnxPuguofgzOa6CpKVZUW913mndLrbXGd23P-yLtDQ0dZ2-u-yqNuH4hEPJH47bQMjLbpafCWIQZvYTCdwVhNPUAMySmEZZAFm-M6L5YJXzUXXX1uz-ftafxYvetxuN_Cz4i8z0CzwBU-xIEhW-oG-MkP8ewBVhPYrx2C3T-eYpYj
   likes  : 42667  comments: 168

New cover images ready to embed: 1


Run Step 2. Confirm the post count and the caption/image sample look right; you should see a mix of `[photo]` and `[video]` entries. With `MAX_PAGES = 100` a deep, video-heavy profile like `@apple` can yield ~1,000+ cover images and bill up to ~100 SerpApi credits (the [free tier](https://serpapi.com/plan) is 250/month). Lower `MAX_PAGES` if you just want a quick sample. Then Step 3 downloads each cover image and embeds it with Jina.

> Note: Instagram CDN URLs expire. The `display_url` / `serpapi_display_url` links are time-limited. That's why Step 3 downloads and embeds each image immediately rather than storing a URL to fetch and embed later.


## Step 3: Embed the Images With Jina v5 Omni

[`jina-embeddings-v5-omni-small`](https://jina.ai/models/jina-embeddings-v5-omni-small/) maps images and text into one shared vector space, so a text query like "a dog wearing sunglasses" retrieves matching images directly, no caption needed at search time.

We embed from Python by calling the [Jina Embeddings API](https://jina.ai/embeddings/): download each cover image, base64-encode it, and send it with `task="retrieval.passage"` (documents). At search time we embed the text query with `task="retrieval.query"`, the asymmetric document/query pair the Jina API provides for retrieval. Both return 1024-d vectors in the same space.

We test on 3 images first to confirm the vector length, then embed the rest.


In [8]:
import os
import base64
from io import BytesIO

import requests
from PIL import Image

os.makedirs(IMAGES_DIR, exist_ok=True)


def download_image(post, timeout=30):
    """Download a post's cover image, save under IMAGES_DIR, return the path.

    Skips the download if we already have the file locally (saves bandwidth + disk churn on
    re-runs). CDN URLs expire, so we fetch immediately when the file is missing.
    """
    path = os.path.join(IMAGES_DIR, f"{post['shortcode']}.jpg")
    if os.path.exists(path):
        return path
    # Prefer SerpApi's proxied URL: it's more stable than the raw Instagram CDN link.
    url = post.get("serpapi_display_url") or post.get("display_url")
    resp = requests.get(url, timeout=timeout)
    resp.raise_for_status()
    img = Image.open(BytesIO(resp.content)).convert("RGB")
    img.save(path, "JPEG", quality=90)
    return path


def image_to_b64(path):
    """Read a local JPEG and return its raw base64 string."""
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("ascii")


def _jina_embed(images_b64=None, texts=None, task="retrieval.passage"):
    """Call the Jina API. v5 omni takes a single `input` list mixing item types:
       text  -> {"text": "a dog wearing sunglasses"}
       image -> {"image": "data:image/jpeg;base64,..."}
    `task` is retrieval.passage for documents (images) and retrieval.query for the search text."""
    if images_b64:
        inputs = [{"image": f"data:image/jpeg;base64,{b}"} for b in images_b64]
    else:
        inputs = [{"text": t} for t in texts]
    payload = {"model": JINA_MODEL, "task": task, "dimensions": EMBED_DIM, "input": inputs}
    r = requests.post(JINA_URL, headers=jina_headers, json=payload, timeout=120)
    r.raise_for_status()
    data = sorted(r.json()["data"], key=lambda d: d["index"])
    return [d["embedding"] for d in data]


def embed_images(images_b64):
    """Embed a batch of base64 images as document vectors."""
    return _jina_embed(images_b64=images_b64, task="retrieval.passage")


def embed_query(text):
    """Embed a search string into the same space as the images."""
    return _jina_embed(texts=[text], task="retrieval.query")[0]

In [9]:
# Test on up to 3 images first: confirm the response shape and vector length before scaling.
if posts:
    test_b64 = [image_to_b64(download_image(p)) for p in posts[:3]]
    test_vectors = embed_images(test_b64)
    print(f"Embedded {len(test_vectors)} images")
    print(f"Vector length: {len(test_vectors[0])}  (expected {EMBED_DIM})")
else:
    print("No new posts to embed. The index is already current for this profile. Skip to Step 5.")

Embedded 1 images
Vector length: 1024  (expected 1024)


If the vector length is `1024` and you got one vector per image, embed them all. The next cell downloads the cover images concurrently (a thread pool, the real speed win for big batches; cached files are skipped), then embeds them in batches and builds one `records` list, each record holds the metadata we'll index plus its `embedding`.

> Why threads and not SerpApi `async`? The downloads are independent HTTP GETs, so concurrency helps a lot. [SerpApi async](https://serpapi.com/search-api#api-parameters-serpapi-parameters-async) is for many independent searches; it can't parallelize one profile's `next_page_token` chain, so it wouldn't speed up our fetch.
>
> The [Jina Embeddings API](https://jina.ai/embeddings/) free tier (with an API key) allows 100 RPM / 100K TPM, so a few hundred images finish in well under a minute; the batch loop only backs off if it brushes a 429.

In [10]:
import time
from concurrent.futures import ThreadPoolExecutor

DOWNLOAD_WORKERS = 12  # concurrent image downloads, the real speed win for big batches


def embed_all(posts, batch_size=EMBED_BATCH, max_retries=4):
    """Download cover images concurrently (cached ones are skipped), then embed in batches.

    Returns the records list (metadata + vector). Only the posts passed in are embedded, so
    on a re-run we spend Jina tokens on genuinely new images only.
    """
    if not posts:
        return []

    # 1) Download all images concurrently. They're independent HTTP GETs to a CDN, so a thread
    #    pool collapses ~hundreds of serial round-trips into a few waves. (This is plain HTTP
    #    concurrency, NOT SerpApi async, which can't parallelize the sequential page cursor.)
    def _download(p):
        try:
            return p, download_image(p)
        except Exception as e:
            print(f"  skip {p.get('shortcode')}: {e}")
            return p, None

    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        downloaded = [(p, path) for p, path in pool.map(_download, posts) if path]
    print(f"  downloaded/cached {len(downloaded)}/{len(posts)} images")

    # 2) Embed in batches (Jina batches internally; light retry only if we brush a 429).
    records = []
    for start in range(0, len(downloaded), batch_size):
        chunk = downloaded[start:start + batch_size]
        b64s = [image_to_b64(path) for _, path in chunk]

        for attempt in range(1, max_retries + 1):
            try:
                vectors = embed_images(b64s)
                break
            except requests.HTTPError as e:
                if attempt == max_retries or e.response.status_code != 429:
                    raise
                wait = 5 * attempt
                print(f"    rate limited, waiting {wait}s…")
                time.sleep(wait)

        for (p, path), vec in zip(chunk, vectors):
            records.append({
                "shortcode": p["shortcode"],
                "post_url": f"https://www.instagram.com/p/{p['shortcode']}/",
                "image_url": p.get("serpapi_display_url") or p.get("display_url"),
                "thumbnail_url": p.get("serpapi_thumbnail_src"),
                "image_path": path,
                "caption": get_caption(p),
                "username": PROFILE_USERNAME,
                "is_video": bool(p.get("is_video")),
                # SerpApi fills liked_by_count only for the newest ~page of posts, while
                # media_preview_likes_count is present on EVERY post (identical when both exist),
                # so we fall back to it for complete like coverage across the whole feed.
                "liked_by_count": p.get("liked_by_count") or p.get("media_preview_likes_count"),
                "comments_count": p.get("comments_count"),
                "embedding": vec,
            })
        print(f"  embedded {len(records)}/{len(downloaded)}")

    return records


records = embed_all(posts)
if records:
    print(f"\nDone: {len(records)} new cover images embedded ({len(records[0]['embedding'])}-d vectors).")
else:
    print("\nNothing new to embed. The index is already up to date for this profile.")

  downloaded/cached 1/1 images


  embedded 1/1

Done: 1 new cover images embedded (1024-d vectors).


Run Step 3. Run the test cell (vector length `1024`), then the full embed cell. You now have a `records` list, metadata + a 1024-d `embedding` each, and images cached in `images/`.

Next, Step 4 creates the Elasticsearch index ([`dense_vector`](https://www.elastic.co/docs/solutions/search/vector/dense-vector), 1024 dims, cosine) and bulk-indexes these records. Make sure `start-local` is running and `ES_API_KEY` is in `.env`.


## Step 4: Index the Vectors in Elasticsearch

We create the index once with a [`dense_vector`](https://www.elastic.co/docs/solutions/search/vector/dense-vector) field (1024 dims, `cosine`, Elasticsearch [handles normalization internally](https://www.elastic.co/docs/reference/elasticsearch/mapping-reference/dense-vector) during comparison) plus the metadata fields, then bulk-load the new records. At this scale (hundreds to low thousands of vectors) we set `index_options: {"type": "flat"}` for exact, full-precision kNN; the 9.x default is an approximate index (`bbq_hnsw`, or `bbq_disk` on 9.4 under a qualifying license), designed for millions of vectors. `dense_vector` + kNN run on Elasticsearch's free [Basic license](https://www.elastic.co/subscriptions), so this keeps working after the `start-local` trial expires.

The index is not dropped on re-runs; we upsert by `shortcode`, so the same profile accumulates only its new posts, and different profiles can coexist in one index (each row carries its `username`). The `caption` is stored as `text` + a `keyword` sub-field, ready if you later want keyword or hybrid search.

In [11]:
# Create the index once (with a dense_vector mapping). We DON'T drop it on re-runs. Step 2
# only brings new posts, so we append/upsert into the existing index. Delete it by hand if you
# ever want to rebuild from scratch:  es.indices.delete(index=ES_INDEX)
if not es.indices.exists(index=ES_INDEX):
    es.indices.create(
        index=ES_INDEX,
        mappings={
            "properties": {
                # cosine: ES normalizes vectors to unit length at index time, then compares with
                # dot_product internally (more efficient).
                # index_options "flat" = exact, full-precision kNN, right at this scale (~100s of
                # vectors). The 9.x default is approximate (bbq_hnsw, or bbq_disk on 9.4), built
                # for millions, so we opt into flat for exact results here.
                "embedding": {
                    "type": "dense_vector",
                    "dims": EMBED_DIM,
                    "similarity": "cosine",
                    "index": True,
                    "index_options": {"type": "flat"},
                },
                # caption as text + a keyword sub-field, ready for keyword / hybrid search later.
                "caption":        {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
                "shortcode":      {"type": "keyword"},
                "post_url":       {"type": "keyword"},
                "image_url":      {"type": "keyword"},
                "thumbnail_url":  {"type": "keyword"},
                "image_path":     {"type": "keyword"},
                "username":       {"type": "keyword"},
                "is_video":       {"type": "boolean"},
                "liked_by_count": {"type": "long"},
                "comments_count": {"type": "long"},
            }
        },
    )
    print(f"Created index '{ES_INDEX}' (dense_vector {EMBED_DIM}d, cosine, exact/flat kNN).")
else:
    print(f"Index '{ES_INDEX}' already exists, appending to it.")

Index 'instagram_photos' already exists, appending to it.


In [12]:
from elasticsearch import helpers

# shortcode as the document _id → idempotent upsert (re-indexing a post just overwrites it).
if records:
    actions = [{"_index": ES_INDEX, "_id": r["shortcode"], **r} for r in records]
    indexed, errors = helpers.bulk(es, actions, stats_only=False, raise_on_error=False)
    es.indices.refresh(index=ES_INDEX)
    print(f"Indexed {indexed} new docs ({len(errors)} errors).")
else:
    print("No new records to index.")

total = es.count(index=ES_INDEX, query={"term": {"username": PROFILE_USERNAME}})["count"]
print(f"Index '{ES_INDEX}' now holds {total} images for @{PROFILE_USERNAME}.")

Indexed 1 new docs (0 errors).
Index 'instagram_photos' now holds 580 images for @apple.


Run Step 4. You should see the index created (first time) or "appending to it" (re-runs), and the per-profile image count. Re-running the same profile only adds its newest posts; switching `PROFILE_USERNAME` adds that profile alongside the others. Next, Step 5 runs the actual semantic search.


## Step 5: Search With Plain Language

We embed `QUERY` into the same shared space and run an Elasticsearch kNN search over the `embedding` field through the [Retrievers API](https://www.elastic.co/docs/solutions/search/retrievers-overview), the current way Elasticsearch expresses vector (and hybrid) search. Results come back ranked by cosine similarity.

The results show only the score and the image, never the caption, on purpose: it makes clear the ranking comes from the picture, not the post text. (A query like `"Samoyed"` surfacing a Samoyed video frame is the proof.)

The first cell defines a small `search()` helper and runs your `QUERY`. The cell after it is your playground: change the text, press Shift+Enter, and you get fresh results instantly; the images are already embedded and indexed, so each new search only embeds the query string (no re-indexing, no extra SerpApi credits).


In [13]:
from IPython.display import Image as IPyImage, display

TOP_K = 5


def search(query, k=TOP_K, username=None, show=True):
    """Embed the text query into the shared space and kNN-search the image vectors.

    Scoped to `username` (defaults to the active PROFILE_USERNAME) via a filter on the knn
    retriever, so results stay within the profile you're exploring even though the index can
    hold several profiles. The Retrievers API (`retriever={"knn": ...}`) is the current
    Elasticsearch way to express vector/hybrid search; `k` is neighbors fetched, `size` is
    hits returned, so we set both to `k`.

    We deliberately show ONLY the score + image, never the caption, so it's unmistakable that
    ranking comes from the picture, not the post text. (Captions are still in each hit's
    `_source` if you want them: e.g. `hits[0]["_source"]["caption"]`.)
    """
    username = username or PROFILE_USERNAME
    query_vector = embed_query(query)
    resp = es.search(
        index=ES_INDEX,
        retriever={
            "knn": {
                "field": "embedding",
                "query_vector": query_vector,
                "k": k,
                "num_candidates": 100,
                "filter": {"term": {"username": username}},
            }
        },
        size=k,
        source_excludes=["embedding"],
    )
    hits = resp["hits"]["hits"]
    if show:
        print(f"Top {len(hits)} image matches for {query!r} in @{username}:")
        for rank, h in enumerate(hits, 1):
            print(f"\n#{rank}  ·  score {h['_score']:.3f}")
            display(IPyImage(filename=h["_source"]["image_path"], width=320))
    return hits


hits = search(QUERY)

Top 5 image matches for 'a dog wearing sunglasses' in @apple:

#1  ·  score 0.737



#2  ·  score 0.679



#3  ·  score 0.672



#4  ·  score 0.669



#5  ·  score 0.668


In [14]:
# ════════════════════════════════════════════════════════════════════
#  🔎  YOUR SEARCH PLAYGROUND
#  Change the text in search("...") below and press Shift+Enter to re-run.
#  Instant. It only embeds the query; the images stay indexed.
#  (Optional: pass k=… to show more/fewer results, e.g. search("a dog", k=8))
# ════════════════════════════════════════════════════════════════════
hits = search("a close-up portrait")

# More ideas to try against @apple (its #ShotoniPhone feed is full of these):
#   "snow-capped mountains"              "horseback riding"
#   "a close-up portrait"                "a cat"
#   "a person at sunset"     "someone holding a phone"

Top 5 image matches for 'a close-up portrait' in @apple:

#1  ·  score 0.625



#2  ·  score 0.624



#3  ·  score 0.621



#4  ·  score 0.621



#5  ·  score 0.619


That's the pipeline. Swap the query in the playground cell to explore, no re-embedding needed. Change `PROFILE_USERNAME` and re-run from Step 2 to add another profile; results stay scoped per profile. Re-running the same profile is cheap and incremental, it fetches only new posts, skips already-downloaded images, and embeds only what's new (saving SerpApi credits, Jina tokens, and disk). The same code runs against local Docker or [Elastic Cloud Serverless](https://www.elastic.co/cloud/serverless), only `.env` changes.


### Production note

This notebook embeds images client-side and searches with a precomputed `query_vector`. That keeps it portable (it runs on any Elasticsearch with no extra setup) and transparent for a tutorial. In production you would usually let Elasticsearch generate the embeddings through an inference endpoint and search with [`query_vector_builder`](https://www.elastic.co/docs/solutions/search/vector/knn) instead of a raw vector, so you query in plain language. Both approaches return the same results; we keep the client-side version here for clarity.